# Grouped TSAM Workflow

This notebook is the interactive client for the grouped Option 1 workflow. The reusable loading, validation, calendar grouping, TSAM aggregation, CSV schema construction, and chart builders live in `tsam_workflows`.


## Method Overview

The workflow clusters days separately by calendar month and by working/non-working status. With the default 5 working-day and 2 non-working-day representative configuration, the year is split into 24 groups and produces 84 representative days.


## Imports And Configuration

The notebook defaults to a lightweight deterministic smoke run so CI can execute it quickly. For the full all-country 5/2 workflow, set `NOTEBOOK_COUNTRIES = None`, `NOTEBOOK_WORKING_CLUSTERS = 5`, and `NOTEBOOK_NON_WORKING_CLUSTERS = 2`, or use the CLI examples in the README.


In [1]:
from pathlib import Path
from typing import Any

import ipywidgets as widgets
from IPython.display import display

from tsam_workflows.charts import (
    build_assignment_calendar_figure,
    build_group_accuracy_figure,
    build_representative_weights_figure,
    style_tsam_figure,
)
from tsam_workflows.config import GroupedWorkflowConfig, default_dataset_specs
from tsam_workflows.grouped import run_grouped_workflow


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "approach_1"
NOTEBOOK_COUNTRIES = ("DE",)
NOTEBOOK_WORKING_CLUSTERS = 1
NOTEBOOK_NON_WORKING_CLUSTERS = 1
RUN_CHART_OUTPUTS = False
RUN_WIDGET_OUTPUTS = False

config = GroupedWorkflowConfig(
    year=2025,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    countries=NOTEBOOK_COUNTRIES,
    working_clusters=NOTEBOOK_WORKING_CLUSTERS,
    non_working_clusters=NOTEBOOK_NON_WORKING_CLUSTERS,
    cluster_method="hierarchical",
)
dataset_specs = default_dataset_specs(DATA_DIR)


## Run Workflow

`run_grouped_workflow` returns the three persisted tables, per-group TSAM results, coverage diagnostics, group accuracy, and the country/feature lookup used by the notebook widgets.


In [3]:
result = run_grouped_workflow(config, dataset_specs)


In [4]:
result.dataset_coverage


,dataset,country_count,countries,missing_from_union
0,demand,34,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...","UA, XK"
1,solar_cf,36,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...",-
2,onwind_cf,36,"AL, AT, BA, BE, BG, CH, CZ, DE, DK, EE, ES, FI...",-
3,ror_cf,34,"AL, AT, BA, BE, BG, CH, CZ, DE, EE, ES, FI, FR...","DK, UA"
4,hydro_inflow,30,"AL, AT, BA, BE, BG, CH, CZ, DE, ES, FI, FR, GR...","DK, EE, LT, LV, NL, SK"


In [5]:
summary = {
    "groups": len(result.tsam_results_by_group),
    "representative_days": len(result.representative_days),
    "day_assignments": len(result.day_assignments_df),
    "reduced_hourly_rows": len(result.reduced_hourly_df),
    "selected_countries": ", ".join(result.selected_countries),
}
summary


{'groups': 24,
 'representative_days': 24,
 'day_assignments': 365,
 'reduced_hourly_rows': 576,
 'selected_countries': 'DE'}

## Output Tables


In [6]:
result.representative_days.head()


,selected_medoid_date,representative_id,group_id,month,day_type,cluster_id,cluster_weight
0,2025-01-12,2025_1_non-working_c0,2025_1_non-working,1,non-working,0,8
1,2025-01-28,2025_1_working_c0,2025_1_working,1,working,0,23
2,2025-02-09,2025_2_non-working_c0,2025_2_non-working,2,non-working,0,8
3,2025-02-20,2025_2_working_c0,2025_2_working,2,working,0,20
4,2025-03-27,2025_3_working_c0,2025_3_working,3,working,0,21


In [7]:
result.day_assignments_df.head()


,cluster_id,cluster_weight,group_id,representative_id
date,,,,
2025-01-01,0,23,2025_1_working,2025_1_working_c0
2025-01-02,0,23,2025_1_working,2025_1_working_c0
2025-01-03,0,23,2025_1_working,2025_1_working_c0
2025-01-04,0,8,2025_1_non-working,2025_1_non-working_c0
2025-01-05,0,8,2025_1_non-working,2025_1_non-working_c0


In [8]:
result.reduced_hourly_df.head()


,DE_demand_2025,DE_solar_2025,DE_onwind_2025,DE_ror_2025,DE_hydro_2025,date,month,day_of_month,weekday,day_type,group_id,group_day_number,cluster_id,cluster_weight,representative_id
snapshot,,,,,,,,,,,,,,,
2025-01-12 00:00:00,35834,0.0,0.326822,0.708762,69,2025-01-12,1,12,Sunday,non-working,2025_1_non-working,4,0,8,2025_1_non-working_c0
2025-01-12 01:00:00,34660,0.0,0.320197,0.703389,69,2025-01-12,1,12,Sunday,non-working,2025_1_non-working,4,0,8,2025_1_non-working_c0
2025-01-12 02:00:00,33905,0.0,0.300183,0.712946,69,2025-01-12,1,12,Sunday,non-working,2025_1_non-working,4,0,8,2025_1_non-working_c0
2025-01-12 03:00:00,33645,0.0,0.290100,0.700891,69,2025-01-12,1,12,Sunday,non-working,2025_1_non-working,4,0,8,2025_1_non-working_c0
2025-01-12 04:00:00,33844,0.0,0.287775,0.700194,69,2025-01-12,1,12,Sunday,non-working,2025_1_non-working,4,0,8,2025_1_non-working_c0


## Summary Charts


In [9]:
if RUN_CHART_OUTPUTS:
    build_assignment_calendar_figure(result).show(config={"responsive": True})
else:
    "Assignment calendar rendering skipped. Set RUN_CHART_OUTPUTS = True to display it."


In [10]:
if RUN_CHART_OUTPUTS:
    build_representative_weights_figure(result).show(config={"responsive": True})
else:
    "Representative weight rendering skipped. Set RUN_CHART_OUTPUTS = True to display it."


In [11]:
if RUN_CHART_OUTPUTS:
    build_group_accuracy_figure(result).show(config={"responsive": True})
else:
    "Group accuracy rendering skipped. Set RUN_CHART_OUTPUTS = True to display it."


## Group-Level TSAM Diagnostic Drilldowns

These controls are notebook-only. The CLI exports each selected widget state as a standalone offline HTML file instead of exporting live ipywidgets.


In [12]:
GROUP_OPTIONS = list(result.group_ids)
COUNTRY_OPTIONS = sorted(result.feature_columns_by_country_and_group)
DEFAULT_COUNTRY = "DE" if "DE" in COUNTRY_OPTIONS else COUNTRY_OPTIONS[0]


def feature_group_options(country: str) -> list[str]:
    return list(result.feature_columns_by_country_and_group[country])


def make_group_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=GROUP_OPTIONS,
        description="Group:",
        layout=widgets.Layout(width="360px"),
    )


def make_country_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=COUNTRY_OPTIONS,
        value=DEFAULT_COUNTRY,
        description="Country:",
        layout=widgets.Layout(width="220px"),
    )


def make_feature_group_dropdown(country: str) -> widgets.Dropdown:
    return widgets.Dropdown(
        options=feature_group_options(country),
        description="Feature:",
        layout=widgets.Layout(width="260px"),
    )


def selected_columns(country: str, feature_group: str) -> list[str]:
    return result.feature_columns_by_country_and_group[country][feature_group]


In [13]:
def display_group_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    output = widgets.interactive_output(plot_function, {"group_id": group_dropdown})
    display(widgets.VBox([widgets.HBox([group_dropdown]), output]))


def display_group_feature_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    country_dropdown = make_country_dropdown()
    feature_group_dropdown = make_feature_group_dropdown(country_dropdown.value)

    def update_feature_groups(change: dict[str, Any]) -> None:
        selected = feature_group_dropdown.value
        options = feature_group_options(change["new"])
        feature_group_dropdown.options = options
        feature_group_dropdown.value = selected if selected in options else options[0]

    country_dropdown.observe(update_feature_groups, names="value")
    output = widgets.interactive_output(
        plot_function,
        {
            "group_id": group_dropdown,
            "country": country_dropdown,
            "feature_group": feature_group_dropdown,
        },
    )
    display(widgets.VBox([widgets.HBox([group_dropdown, country_dropdown, feature_group_dropdown]), output]))


In [14]:
def show_cluster_representatives(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_representatives(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Cluster representative profiles: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_feature_plot(show_cluster_representatives)
else:
    "Representative-profile widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


In [15]:
def show_cluster_members(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_members(
        columns=selected_columns(country, feature_group),
        slider="cluster",
        title="",
    )
    style_tsam_figure(fig, f"Cluster members: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_feature_plot(show_cluster_members)
else:
    "Cluster-member widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


In [16]:
def show_cluster_weights(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_weights(title="")
    style_tsam_figure(fig, f"Cluster weights: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_plot(show_cluster_weights)
else:
    "Cluster-weight widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


In [17]:
def show_cluster_accuracy(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.accuracy(title="")
    style_tsam_figure(fig, f"Cluster accuracy: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_plot(show_cluster_accuracy)
else:
    "Cluster-accuracy widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


In [18]:
def show_full_resolution_comparison(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.compare(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Original vs reconstructed: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_feature_plot(show_full_resolution_comparison)
else:
    "Comparison widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


In [19]:
def show_cluster_residuals(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.residuals(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Residuals: {group_id}").show(config={"responsive": True})


if RUN_WIDGET_OUTPUTS:
    display_group_feature_plot(show_cluster_residuals)
else:
    "Residual widget skipped. Set RUN_WIDGET_OUTPUTS = True to display it."


## Optional CSV Export

The CLI is the preferred reproducible export path because it stages outputs before publishing. Set `SAVE_OUTPUTS = True` when you want this notebook to write the same CSV filenames during an interactive run.


In [20]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (result.reduced_hourly_df.reset_index()).to_csv(
        OUTPUT_DIR / "reduced_hourly_df.csv",
        index=False,
    )
    result.representative_days.to_csv(OUTPUT_DIR / "representative_days.csv", index=False)
    (result.day_assignments_df.reset_index()).to_csv(
        OUTPUT_DIR / "day_assignments_df.csv",
        index=False,
    )
    export_summary = {
        "reduced_hourly_df.csv": len(result.reduced_hourly_df),
        "representative_days.csv": len(result.representative_days),
        "day_assignments_df.csv": len(result.day_assignments_df),
    }
else:
    export_summary = {"csv_export": "skipped", "output_dir": str(OUTPUT_DIR)}

export_summary


{'csv_export': 'skipped',
 'output_dir': '/Users/koval/dev/tsam-notebook-workflows/outputs/approach_1'}